# **Tutorial for Querying Bluesky Data with ClickHouse**

## Introduction

This notebook will explain how to connect to the Diderot Bluesky ClickHouse database and querying each ingested API table. From Chengyi's understanding, this is a mirror of the Bluesky content from mid August 2026.

**Prerequisites**

* Conda environment from ``environment.yml (diderot-clinic)``
* A ``.env`` file in the repo root with ``CLICKHOUSE_USERNAME`` and ``CLICKHOUSE_PASSWORD`` provided by Henri

## **Table of Contents**

| Section | Name | Description |
|---------|------|-------------|
| 0 | [Connect and Inspect](#0-connect-and-inspect) | Connect to database and inspect data. |
| 1 | [Posts](#1-posts-records_app_bsky_feed_post) | Post records |
| 2 | [Reposts](#2-posts-records_app_bsky_feed_repost) | Repost records |
| 3 | [Likes](#3-likes-records_app_bsky_feed_like) | Like records |
| 4 | [Follows](#4-follows-records_app_bsky_graph_follow) | Follow records |
| 5 | [Profiles](#5-profiles-records_app_bsky_actor_profile) | Profile records |
| 6 | [Glossary](#6-glossary) | Glossary of all API endpoints |

## 0. Connect and Inspect

We first connect to the database through a clickhouse client.

In [27]:
import clickhouse_connect # clickhouse API
import os # environment variables
import pandas as pd # for data visualization
from dotenv import load_dotenv # load .env file

load_dotenv()

client = clickhouse_connect.get_client(
    host="ch.bsky.diderot.app",
    port=8443,
    username=os.getenv("CLICKHOUSE_USERNAME"),
    password=os.getenv("CLICKHOUSE_PASSWORD"),
    secure=True,
)


def run(sql: str):
    """Run a query and return (column_names, rows)."""
    result = client.query(sql)
    return result.column_names, result.result_rows


def show(sql: str, max_rows: int = 10000):
    """Run a query and display the result as a pandas DataFrame."""
    cols, rows = run(sql)
    df = pd.DataFrame(rows, columns=cols)
    display(df.head(max_rows))

Now we can view the tables that live in the ``bluesky_ingest`` database provided by Diderot.

In [28]:
show("SHOW TABLES FROM bluesky_ingest")

,name
0,records
1,records_app_bsky_actor_profile
2,records_app_bsky_actor_status
3,records_app_bsky_feed_generator
4,records_app_bsky_feed_generator__description_f...
5,records_app_bsky_feed_like
6,records_app_bsky_feed_post
7,records_app_bsky_feed_post__entities
8,records_app_bsky_feed_post__facets
9,records_app_bsky_feed_postgate


## 1. Posts (`records_app_bsky_feed_post`)

Approx. size: ~2.8B rows (please use `LIMIT`s and filters; avoid unbounded `GROUP BY` or full-table scans)

This table contains ingested Bluesky post records (creates, updates, and deletes). Each row is one firehose event for an app.bsky.feed.post record.

### 1.1 Inspect the schema

You could run the snippet below to see the columns in the ingested post records.

In [29]:
# show("DESCRIBE bluesky_ingest.records_app_bsky_feed_post")

Alternatively, here is the summary:

| Column | Type | Description |
| ------ | ---- | ----------- |
| `tid_us` | `UInt64` | Ingest timestamp (microseconds) |
| `did` | `String` | Author DID |
| `rkey` | `String` | Record key (post id within the repo) |
| `cid` | `String` | Content ID of the record |
| `action` | `Enum8('create','update','delete')` | Firehose action |
| `rev` | `String` | Repo revision |
| `live` | `Bool` | Whether the record is currently live |
| `text` | `Nullable(String)` | Post text |
| `created_at` | `Nullable(String)` | Author-declared creation time (ISO-8601) |
| `embed_type` | `LowCardinality(Nullable(String))` | Embed lexicon type, e.g. `app.bsky.embed.images` |
| `embed_record_uri` / `embed_record_cid` | `Nullable(String)` | Quote / record embed |
| `embed_external_*` | various | Link card embed fields |
| `embed_video_*` | various | Video embed fields |
| `embed_image_*` | arrays | Image CIDs, alts, aspect ratios |
| `labels` | `Array(String)` | Self-labels on the post |
| `langs` | `Array(String)` | Declared language codes |
| `reply_root_uri` / `reply_root_cid` | `Nullable(String)` | Root of the thread (if reply) |
| `reply_parent_uri` / `reply_parent_cid` | `Nullable(String)` | Parent post (if reply) |
| `tags` | `Array(String)` | Hashtag-style tags |
| `event_time` | `DateTime64(6)` | Event time in ClickHouse |

### 1.2 First Query

With the snippet below, you can make your first databse query to show posts! Useful columns to start with would be `did` (author ID), `rkey` (record key, i.e. post ID), `text` (the text of the post itself), `created_at` (post creation time), `action` (what was being done to the post), `live` (whether the record is currently live, i.e. live at the time of the mirror), and `embed_type` (embeded media such as images/videos/audio etc.).

In [30]:
show(
    """
    SELECT did, rkey, action, live, text, created_at, embed_type, langs, tags
    FROM bluesky_ingest.records_app_bsky_feed_post
    LIMIT 5
    """
)

,did,rkey,action,live,text,created_at,embed_type,langs,tags
0,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jj4ws2h,create,False,a,2024-07-17T13:52:47.000Z,NaN,[],[]
1,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jj4wtjo,create,False,b,2024-07-17T13:52:47.001Z,NaN,[],[]
2,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb3267,create,False,c,2024-07-17T13:53:24.000Z,NaN,[],[]
3,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb33on,create,False,d,2024-07-17T13:53:24.001Z,NaN,[],[]
4,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantkoraasfx,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:04:08.1736565+00:00,app.bsky.embed.images,[],[]


> **IMPORTANT**: Remember to always limit your queries to prevent timeout! We have a maximum query time of 120s.

### 1.3 Sampling posts with text

As the above query results suggest, some posts have been deleted or have useless information. To filter for useful posts, we may add a WHERE field in the SQL:

In [31]:
show(
    """
    SELECT did, rkey, action, live, text, created_at, embed_type, langs, tags
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE text IS NOT NULL AND length(text) > 10
    LIMIT 5
    """
)

,did,rkey,action,live,text,created_at,embed_type,langs,tags
0,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantkoraasfx,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:04:08.1736565+00:00,app.bsky.embed.images,[],[]
1,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantoevwdxu5,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:06:12.0594507+00:00,app.bsky.embed.images,[],[]
2,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranu625xochs,create,False,https://localhost:5173/did:plc:2edipcwcjiezjta...,2024-11-11T09:14:57.6594223+00:00,app.bsky.embed.images,[],[]
3,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvf5cjjqgr,create,False,https://localhost:5173/did:plc:2edipcwcjiezjta...,2024-11-11T09:36:49.5745663+00:00,app.bsky.embed.images,[],[]
4,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvlk5udvbp,create,False,https://example.com/did:plc:2edipcwcjiezjtanjs...,2024-11-11T09:40:24.3800382+00:00,app.bsky.embed.images,[],[]


### 1.4 Posts by one author (`did`)

Each user has a unique `did`. To obtain a user's `did`, you may run the following function:

In [32]:
import urllib.request, json
def get_did(handle: str) -> str:
    """Get the DID of a given handle."""  
    url = f"https://public.api.bsky.app/xrpc/com.atproto.identity.resolveHandle?handle={handle}"
    return json.load(urllib.request.urlopen(url))["did"]

For example, we can run this on Alexandria Ocasio-Cortez's bluesky handle:

In [33]:
aoc_handle = "aoc.bsky.social"
aoc_did = get_did(aoc_handle)
print(aoc_did)

did:plc:p7gxyfr5vii5ntpwo7f6dhe2


We can then filter posts made by her:

In [34]:
show(
    f"""
    SELECT did, rkey, text, created_at, action, live
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE did = '{aoc_did}'
    LIMIT 5
    """
)

,did,rkey,text,created_at,action,live
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juexmwyc3o2h,Hi there! Is this thing on? 🎤,2023-04-27T20:42:40.540Z,create,False
1,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf44f52li2k,Ok! Here is Deco at the airport from a few wee...,2023-04-27T22:02:53.788Z,create,False
2,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf52klwc72i,Here you go!,2023-04-27T22:19:46.107Z,create,False
3,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf56kh5ad2f,This is so sweet 🥹 thank you,2023-04-27T22:22:00.161Z,create,False
4,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf5lltngs2j,I just got here and it’s already more enjoyabl...,2023-04-27T22:29:17.738Z,create,False


Obviously it would also be helpful to find the handle of someone given their `did`, which you can do with this function:

In [35]:
def get_handle(did: str) -> str:
    """Get the handle of a given DID."""
    url = f"https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile?actor={did}"
    return json.load(urllib.request.urlopen(url))["handle"]

print(get_handle(aoc_did))

aoc.bsky.social


### 1.5 Replies only

Replies set `reply_parent_uri` (and usually `reply_root_uri`). URIs look like:
`at://did:plc:.../app.bsky.feed.post/<rkey>`.

In [36]:
show(
    """
    SELECT did, rkey, text, created_at, reply_root_uri, reply_parent_uri
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE reply_parent_uri IS NOT NULL AND text IS NOT NULL
    LIMIT 5
    """
)

,did,rkey,text,created_at,reply_root_uri,reply_parent_uri
0,did:plc:6al5t6ei6z6p4wzpmhpmwmdw,3jmxyghkxwq2p,pretty great costumes ngl,2023-01-23T14:55:58.472Z,at://did:plc:oky5czdrnfjpqslsw2a5iclo/app.bsky...,at://did:plc:oky5czdrnfjpqslsw2a5iclo/app.bsky...
1,did:plc:6al5t6ei6z6p4wzpmhpmwmdw,3jmy5ik7wpa2p,hi!,2023-01-23T16:26:37.086Z,at://did:plc:rg2lsu5lmbw5vou4ec5lwctv/app.bsky...,at://did:plc:rg2lsu5lmbw5vou4ec5lwctv/app.bsky...
2,did:plc:6al5t6ei6z6p4wzpmhpmwmdw,3jmy5qs3mwy2p,obsidian over notes?,2023-01-23T16:31:13.335Z,at://did:plc:w5lm6utpdvsg5spyybh76sae/app.bsky...,at://did:plc:vpkhqolt662uhesyj6nxm7ys/app.bsky...
3,did:plc:6al5t6ei6z6p4wzpmhpmwmdw,3jmzlesi2oo2p,gotta give it another try - works well offline...,2023-01-24T06:07:43.661Z,at://did:plc:w5lm6utpdvsg5spyybh76sae/app.bsky...,at://did:plc:vpkhqolt662uhesyj6nxm7ys/app.bsky...
4,did:plc:6al5t6ei6z6p4wzpmhpmwmdw,3jmzlgieaj62p,legit,2023-01-24T06:08:40.238Z,at://did:plc:ymdx5ssbzyhxttf6pkongcjw/app.bsky...,at://did:plc:ymdx5ssbzyhxttf6pkongcjw/app.bsky...


### 1.6 Posts with embeds

`embed_type` is a Bluesky embed lexicon, e.g. `app.bsky.embed.images` or `app.bsky.embed.external`.

In [37]:
show(
    """
    SELECT did, rkey, text, embed_type, embed_external_uri, embed_external_title,
           length(embed_image_cids) AS n_images
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE embed_type IS NOT NULL
    LIMIT 5
    """
)

,did,rkey,text,embed_type,embed_external_uri,embed_external_title,n_images
0,did:plc:65gobtxqtmd4qt2vxjpn37ts,32xuvjmwzy22a,Yoooooo youtu.be/jzTIw8hPVRU?...,app.bsky.embed.external,https://youtu.be/jzTIw8hPVRU?si=6IKnkXFnY43LT5rX,Official FIFA 07 Trailer,0
1,did:plc:hbfkgnstjlgkaa57pcjkaf25,3jmlwvyzqen2h,NaN,app.bsky.embed.images,NaN,NaN,1
2,did:plc:hbfkgnstjlgkaa57pcjkaf25,3jmp7wfjhtv2p,NaN,app.bsky.embed.images,NaN,NaN,1
3,did:plc:hbfkgnstjlgkaa57pcjkaf25,3jmp7xsbvre2e,NaN,app.bsky.embed.images,NaN,NaN,1
4,did:plc:hbfkgnstjlgkaa57pcjkaf25,3jmpa46y43e2e,BlueSkys motto?? @dan.bsky.social thoughts??,app.bsky.embed.images,NaN,NaN,1


### 1.7 Simple text search

`ILIKE` works, but can be slow on the full table. Keep `LIMIT` small to avoid timeout.

In [38]:
show(
    """
    SELECT did, rkey, text, created_at
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE text ILIKE '%meow%'
    LIMIT 10
    """
)

,did,rkey,text,created_at
0,did:plc:c54vmaxhquiocmettp5zgvka,3jrmrxrlvz22i,Meowchelin star chef,2023-03-23T19:07:04.640Z
1,did:plc:ik2tenugs6vlfbj4pmepzqpp,3ju52yfo5ia2a,Meow world,2023-04-24T17:21:28.421Z
2,did:plc:g22wrkiovld7mymdemad34t3,3jurbbvlqdx2v,Meow,2023-05-02T18:07:24.432Z
3,did:plc:244dhc5kw7asrhd7i4li7khx,3jt3xf3qzcd22,New and exciting homeowner bit where the tempe...,2023-04-11T13:19:07.917Z
4,did:plc:246htihk27sb6zbixesib7cz,3jue32gxxgd2n,gmeow,2023-04-27T12:11:15.006Z
5,did:plc:pwf67cusdbxrptb4d463rbj7,3jukmolaasw2s,does anyone want to give a beautiful princess ...,2023-04-30T02:42:42.628Z
6,did:plc:mou6t7mhh4ddhcake2a4d53r,3jty2hpuelt2u,なるほど Meow安寺というのもあったかニャ๑(ΦωΦ)๑,2023-04-22T17:28:49.768Z
7,did:plc:22rlbadfyb22mvwexddamubz,3jxcvqcflbd2v,Meow,2023-06-04T05:19:01.118Z
8,did:plc:gyztjuvo7gud3guhqkycseny,3jxzxpzg2y22n,I have turned into that person who takes lots ...,2023-06-13T09:25:52.959Z
9,did:plc:24ggxmnj7734oecl6ne7ja3k,3jy5loikuw72m,"throw licorice into the void, meow",2023-06-14T20:00:56.067Z


## 2. Reposts (`records_app_bsky_feed_repost`)

Approx. size: ~2.4B rows (please use `LIMIT`s and filters; avoid unbounded `GROUP BY` or full-table scans)

This table contains ingested Bluesky repost records (creates, updates, and deletes). Each row is one firehose event for an [`app.bsky.feed.repost`](https://docs.bsky.app/docs/api/app-bsky-feed-repost) record, i.e. when an account reposts someone else's post. The reposted post is identified by `subject_uri` / `subject_cid`. Prefer filtering on `did`; filtering only on `subject_uri` can time out.

### 2.1 Inspect the schema

You could run the snippet below to see the columns in the ingested repost records.

In [39]:
# show("DESCRIBE bluesky_ingest.records_app_bsky_feed_repost")

Alternatively, here is the summary:

| Column | Type | Description |
| ------ | ---- | ----------- |
| `tid_us` | `UInt64` | Ingest timestamp (microseconds) |
| `did` | `String` | DID of the account that reposted |
| `rkey` | `String` | Record key of the repost |
| `cid` | `String` | Content ID of the repost record |
| `action` | `Enum8('create','update','delete')` | Firehose action |
| `rev` | `String` | Repo revision |
| `live` | `Bool` | Whether the record is currently live |
| `subject_uri` | `Nullable(String)` | AT-URI of the post being reposted |
| `subject_cid` | `Nullable(String)` | CID of the subject post |
| `created_at` | `Nullable(String)` | Author-declared creation time (ISO-8601) |
| `via_uri` / `via_cid` | `Nullable(String)` | Optional source repost (repost-of-repost) |
| `event_time` | `DateTime64(6)` | Event time in ClickHouse |

### 2.2 First Query

With the snippet below, you can make your first database query to show reposts! Useful columns to start with would be `did` (who reposted), `rkey` (record key), `subject_uri` / `subject_cid` (the original post), `created_at` (repost creation time), `action` (what was being done to the repost), `live` (whether the record is currently live), and optional `via_uri` (repost-of-a-repost, useful for constructing information cascades).

In [20]:
show(
    """
    SELECT did, rkey, action, live, subject_uri, subject_cid, created_at, via_uri
    FROM bluesky_ingest.records_app_bsky_feed_repost
    LIMIT 5
    """
)

,did,rkey,action,live,subject_uri,subject_cid,created_at,via_uri
0,did:plc:25qfmw6msghyj5jrcunikymy,3jpvnfbukvc2m,create,False,at://did:plc:3cij4jaflmx7wmnzvquxfys3/app.bsky...,bafyreig6dk6ymdiqeipwqzbbr4xn7cn6ftv3g4tveqbjt...,2023-03-01T20:48:43.373Z,None
1,did:plc:25qfmw6msghyj5jrcunikymy,3jpxr72gla222,create,False,at://did:plc:xr2ehpmzbody6tgehsp4yubu/app.bsky...,bafyreihme7sttizrc3mxf7ujjs7yewwon3r2fxxmbyvjs...,2023-03-02T17:02:08.502Z,None
2,did:plc:25qfmw6msghyj5jrcunikymy,3jpyeaivjls2p,create,False,at://did:plc:34w2qqfauelc42s57x2mc4dt/app.bsky...,bafyreiam7rbjagp42j62oisryp5ybclnn7a65q4ynxs3c...,2023-03-02T22:42:58.439Z,None
3,did:plc:22ixs7kj3hyjzoksrrivaixs,3jq4ubfzk5k2x,create,False,at://did:plc:22ixs7kj3hyjzoksrrivaixs/app.bsky...,bafyreiayo3wygmpvgzqrujdkxkyv77lls5lwhr2npwq7h...,2023-03-04T17:40:28.494Z,None
4,did:plc:22ixs7kj3hyjzoksrrivaixs,3jq5lu4m2js2e,create,False,at://did:plc:4gkxysoweqvpbgebxrsry6rp/app.bsky...,bafyreiemzk4gf6qrzxeqdqcq7tqcptfitudfmodwopvye...,2023-03-05T00:42:31.845Z,None


### 2.3 Live creates only

`live = true` and `action = 'create'` keeps currently active repost records.

In [21]:
show(
    """
    SELECT did, rkey, subject_uri, subject_cid, created_at
    FROM bluesky_ingest.records_app_bsky_feed_repost
    WHERE live = true AND action = 'create'
    LIMIT 5
    """
)

,did,rkey,subject_uri,subject_cid,created_at
0,did:plc:5r6gsajuu4z6k4svvl5eags4,22222222c6y22,at://did:plc:htguljkxujej746dcydxys2a/app.bsky...,bafyreifva2xwwmtmlit7cmfsm5pt67di5a5j3uxjp7fci...,2026-08-14T22:15:54.213695Z
1,did:plc:5r6gsajuu4z6k4svvl5eags4,22222222c6z22,at://did:plc:htguljkxujej746dcydxys2a/app.bsky...,bafyreie7sx3pg3w74xjk42mhzgjwqgtstldmrgbopkxeu...,2026-08-14T22:17:10.242389Z
2,did:plc:5r6gsajuu4z6k4svvl5eags4,22222222c7222,at://did:plc:htguljkxujej746dcydxys2a/app.bsky...,bafyreiflhsjvzdozvdmokrkwd7kv3licohgdgnwnx7yt6...,2026-08-14T22:22:51.707570Z
3,did:plc:34ynawve37db2zmvdyjxdo42,3mt25gqb2r62j,at://did:plc:ie3apbnprawhixkka5y4yusm/app.bsky...,bafyreigk24y5pjx3gusy723oywwizcyjt65svocnc6jia...,2026-08-14T12:20:42.701Z
4,did:plc:27xg4qrrmynwowtnm25kvmq5,3mt25gr6n7c2x,at://did:plc:5uh7sohpgujgipximrnsfumg/app.bsky...,bafyreie3aybktpkqcldxcnzkt5owjzi5vtexv6wxlch6o...,2026-08-14T12:20:42.994Z


### 2.4 Reposts with `via`

When present, `via_uri` points at another repost record (e.g. `at://did:plc:.../app.bsky.feed.repost/<rkey>`).

In [23]:
show(
    """
    SELECT did, rkey, subject_uri, via_uri, via_cid, created_at
    FROM bluesky_ingest.records_app_bsky_feed_repost
    WHERE via_uri IS NOT NULL
    LIMIT 5
    """
)

,did,rkey,subject_uri,via_uri,via_cid,created_at
0,did:plc:km7x7jb44xz4vtbicjrutkqb,3lqeupa5cn42c,at://did:plc:wxw6earvofc6q3xbz26dew3z/app.bsky...,at://did:plc:hj7c7a26uvhntmmfwmhnwzgu/app.bsky...,bafyreifh43scnxwjsu6cz6uznpqdblr3zenbdihi7axlb...,2025-05-30T09:01:54.237Z
1,did:plc:km7x7jb44xz4vtbicjrutkqb,3lqeupippwx2i,at://did:plc:iphaurh4b7oputpujjo3grao/app.bsky...,at://did:plc:cxkyvdaevgvuot3ud3ubhtof/app.bsky...,bafyreic2lx2iv7gtb4527qkqyizribtl2dolv5acoi6vb...,2025-05-30T09:02:03.233Z
2,did:plc:km7x7jb44xz4vtbicjrutkqb,3lqeuq7smuy2h,at://did:plc:x6j2p7ew43cingyhidckflut/app.bsky...,at://did:plc:x6j2p7ew43cingyhidckflut/app.bsky...,bafyreihaqukpbtsxumb5myhfkux43bk3t5sksgiqohish...,2025-05-30T09:02:27.443Z
3,did:plc:gz334bwmqldnturg6e7vbnm6,3lqbu4xboce2w,at://did:plc:3uxsyzqyom4fru2sqvhkzpfy/app.bsky...,at://did:plc:3uxsyzqyom4fru2sqvhkzpfy/app.bsky...,bafyreigfn7kscfdnuub7yi7vldvqinl5p5gfukl3vpue7...,2025-05-29T04:13:42.123Z
4,did:plc:eb4gabjheyakvfptvd22zmzj,3lqv3eas6dp2n,at://did:plc:az4ntj4224lyl7kphcrlocqt/app.bsky...,at://did:plc:kba5ra5zizxsey2nq4pwnove/app.bsky...,bafyreihb6fxgm3srts5bcv4tnq734re3zx4qctum3ko4m...,2025-06-05T19:43:37.494Z


### 2.5 Reposts by one account (`did`)

Same pattern as posts: filter by author DID for fast exploration. We can reuse the `aoc_did` variable from [section 1.4](#14-posts-by-one-author-did):

In [25]:
show(
    f"""
    SELECT did, rkey, subject_uri, created_at, live, action
    FROM bluesky_ingest.records_app_bsky_feed_repost
    WHERE did = '{aoc_did}'
    LIMIT 5
    """
)

,did,rkey,subject_uri,created_at,live,action
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juh66meetv2m,at://did:plc:53pxyzwrw4zx5ft67czyyjyy/app.bsky...,2023-04-28T17:45:15.277Z,False,create
1,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juhpzsoply2f,at://did:plc:53kudcqj7sizx2eaclc5sicc/app.bsky...,2023-04-28T23:04:41.483Z,False,create
2,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3jupufe24z32i,at://did:plc:222rnvnta2lbl364bog2plxw/app.bsky...,2023-05-02T04:44:01.809Z,False,create
3,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juqsedtv3v25,at://did:plc:64zyeexwq37nx6h7xvke4cow/app.bsky...,2023-05-02T13:40:20.146Z,False,create
4,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juylmjvzwe2m,at://did:plc:qiknc4t5rq7yngvz7g4aezq7/app.bsky...,2023-05-05T16:00:56.687Z,False,create


### 2.6 Retrieve the original post content

A repost row does not store the original post text. It only points at the original via `subject_uri`, which looks like:

`at://did:plc:.../app.bsky.feed.post/<rkey>`

To get the content, parse that URI into the author's `did` and the post's `rkey`, then look them up in `records_app_bsky_feed_post`. Prefer this two-step pattern over joining the full tables — large joins on `subject_uri` can time out or hit memory limits.

First, take one of AOC's reposts and print its `subject_uri`:

In [40]:
_, rows = run(
    f"""
    SELECT subject_uri, created_at
    FROM bluesky_ingest.records_app_bsky_feed_repost
    WHERE did = '{aoc_did}' AND subject_uri IS NOT NULL
    LIMIT 1
    """
)
subject_uri, reposted_at = rows[0]
print("subject_uri:", subject_uri)
print("reposted_at:", reposted_at)

subject_uri: at://did:plc:53pxyzwrw4zx5ft67czyyjyy/app.bsky.feed.post/3juh4g2yelc2q
reposted_at: 2023-04-28T17:45:15.277Z


Parse the AT-URI into author DID and post rkey, then fetch the original post:

In [41]:
parts = subject_uri.split("/")
author_did, post_rkey = parts[2], parts[4]
print("author_did:", author_did)
print("post_rkey:", post_rkey)

show(
    f"""
    SELECT did, rkey, text, created_at, embed_type
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE did = '{author_did}' AND rkey = '{post_rkey}'
    LIMIT 1
    """
)

author_did: did:plc:53pxyzwrw4zx5ft67czyyjyy
post_rkey: 3juh4g2yelc2q


,did,rkey,text,created_at,embed_type
0,did:plc:53pxyzwrw4zx5ft67czyyjyy,3juh4g2yelc2q,Some blueberries for bluesky.\n#WeFeedYou,2023-04-28T17:13:37.975Z,app.bsky.embed.images


## 3. Likes (`records_app_bsky_feed_like`)

## 4. Follows (`records_app_bsky_graph_follow`)

## 5. Profiles (`records_app_bsky_actor_profile`)

## 6. Glossary